In [ ]:
%%capture
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dj_notebook import activate
from django_pandas.io import read_frame

env_file = os.environ["META_ENV"]
reports_folder = Path(os.environ["META_REPORTS_FOLDER"])
analysis_folder = Path(os.environ["META_ANALYSIS_FOLDER"])
pharmacy_folder = Path(os.environ["META_PHARMACY_FOLDER"])
plus = activate(dotenv_file=env_file)
pd.set_option("future.no_silent_downcasting", True)

In [ ]:
from meta_subject.models import BloodResultsIns
from meta_subject.models import Glucose, GlucoseFbg
from edc_pdutils.dataframes import get_subject_visit
from meta_screening.models import SubjectScreening
from clinicedc_constants import YES, NO
from meta_rando.models import RandomizationList
from meta_prn.models import EndOfStudy

In [ ]:
export_to_file = False

In [ ]:
df_screening_orig:pd.DataFrame = read_frame(SubjectScreening.objects.all())

In [ ]:
df_rando = read_frame(RandomizationList.objects.values("subject_identifier", "sid", "assignment", "randomizer_name", "allocated_site").all(), verbose=False)

In [ ]:
df_eos = read_frame(EndOfStudy.objects.all())
df_eos[["subject_identifier", "offstudy_datetime", "last_seen_date", ]]

In [ ]:
df_visit = get_subject_visit(model="meta_subject.subjectvisit")

In [ ]:
df_visit = df_visit.loc[df_visit["reason"] != "missed"].reset_index(drop=True)

df_visit["visit_count"] = df_visit.groupby(by=["subject_identifier"])["subject_identifier"].transform("count")

In [ ]:
df_visit.reason.value_counts()

In [ ]:
df_visit.site_id.value_counts()

In [ ]:
df_screening = (
    df_screening_orig.copy()
    .rename(columns={
        "fbg_value": "fbg1_value",
        "fbg_units": "fbg1_units",
        "repeat_fbg_units": "fbg2_units",
        "converted_fbg_value":"converted_fbg1_value",
        "fasting": "fasting1",
        "repeat_fasting": "fasting2",
        "fasting_duration_delta": "fasting1_duration_delta",
        "repeat_fasting_duration_delta": "fasting2_duration_delta",
        "fbg_datetime": "fbg1_datetime",
        "repeat_fbg_datetime": "fbg2_datetime",


    })
)

In [ ]:
df_screening["source"] = "meta_screening.subjectscreeening"

df_screening["fbg_value"] = np.where(
    df_screening["fbg2_value"].notna(),
    df_screening["fbg2_value"],
    df_screening["fbg1_value"]
)

df_screening["converted_fbg_value"] = np.where(
    df_screening["converted_fbg2_value"].notna(),
    df_screening["converted_fbg2_value"],
    df_screening["converted_fbg1_value"]
)

df_screening["fasting"] = np.where(
    df_screening["fbg2_value"].notna(),
    df_screening["fasting2"],
    df_screening["fasting1"]
)

df_screening["fbg_units"] = np.where(
    df_screening["fbg2_units"].notna(),
    df_screening["fbg2_units"],
    df_screening["fbg1_units"]
)

df_screening["fasting_duration_delta"] = np.where(
    df_screening["fbg2_value"].notna(),
    df_screening["fasting2_duration_delta"],
    df_screening["fasting1_duration_delta"]
)

df_screening["fbg_datetime"] = np.where(
    df_screening["fbg2_datetime"].notna(),
    df_screening["fbg2_datetime"],
    df_screening["fbg1_datetime"]
)

df_screening["fasting_hrs"] = df_screening["fasting_duration_delta"].apply(lambda x: x.total_seconds() / 3600)

df_screening["visit_code"] = 0.0


In [ ]:
df_screening = df_screening.loc[(df_screening["fasting_duration_delta"] >= pd.Timedelta(hours=8.0))][["subject_identifier", "fbg_value", "fbg_units", "fbg_datetime", "fasting_hrs", "visit_code"]]

df_screening = df_screening.loc[df_screening["subject_identifier"].str.startswith("105-")]
df_screening = df_screening.merge(df_visit[["subject_identifier", "baseline_datetime"]].drop_duplicates(), on="subject_identifier", how="left").reset_index(drop=True)

# df_screening

In [ ]:
df_fbg1 = read_frame(Glucose.objects.all())[["subject_visit", "fbg_datetime", "fbg_value", "fbg_units", "fasting_duration_delta"]]

df_fbg2 = read_frame(GlucoseFbg.objects.all())[["subject_visit", "fbg_datetime", "fbg_value", "fbg_units", "fasting_duration_delta"]]

df_fbg1["fasting_hrs"] = np.nan
df_fbg1["fasting_hrs"] = df_fbg1["fasting_duration_delta"].apply(lambda x: x.total_seconds() / 3600)

df_fbg2["fasting_hrs"] = np.nan
df_fbg2["fasting_hrs"] = df_fbg2["fasting_duration_delta"].apply(lambda x: x.total_seconds() / 3600)

df_fbg = pd.concat([
    df_fbg1[~(df_fbg1["fbg_datetime"].isna()) & ~(df_fbg1["fbg_value"].isna()) & (df_fbg1["fasting_hrs"]>=8.0)],
    df_fbg2[~(df_fbg2["fbg_datetime"].isna()) & ~(df_fbg2["fbg_value"].isna()) & (df_fbg2["fasting_hrs"]>=8.0)]],
    ignore_index=True
)

df_fbg = df_fbg.rename(columns={"subject_visit": "subject_visit_id"})

df_fbg = df_fbg.merge(df_visit, on="subject_visit_id", how="left")

df_fbg = df_fbg[["subject_identifier", "baseline_datetime", "fbg_value", "fbg_units", "fbg_datetime", "fasting_hrs", "visit_code"]].reset_index(drop=True)


In [ ]:
df_fbg = pd.concat([df_fbg[["subject_identifier", "baseline_datetime", "fbg_value", "fbg_units", "fbg_datetime", "fasting_hrs", "visit_code"]], df_screening[["subject_identifier", "baseline_datetime","fbg_value", "fbg_units", "fbg_datetime", "fasting_hrs", "visit_code"]]], ignore_index=True)

In [ ]:
df_fbg[df_fbg["subject_identifier"].isna()]

In [ ]:
df_fbg[df_fbg["fbg_datetime"].isna()]


In [ ]:
df_fbg = df_fbg.sort_values(["subject_identifier", "fbg_datetime"]).reset_index(drop=True)
ranks = df_fbg.groupby("subject_identifier")["fbg_datetime"].rank(method="first")
counts = df_fbg.groupby("subject_identifier")["fbg_datetime"].transform("count")

df_fbg["timing"] = np.select([ranks==1, ranks==counts], ["first", "last"], default="middle")

In [ ]:
df_fbg = df_fbg[
    (df_fbg["subject_identifier"].str.startswith("105-")) &
    (df_fbg["timing"].isin(["first", "last"]))
].copy().reset_index(drop=True)

# df_fbg


In [ ]:
# validation
df_fbg = df_fbg.merge(df_visit[["subject_identifier", "visit_count"]].drop_duplicates()
, on="subject_identifier", how="left").reset_index(drop=True)

df_fbg["record_count"] = df_fbg.groupby("subject_identifier").subject_identifier.transform("count")

In [ ]:
df_fbg.subject_identifier.nunique()

In [ ]:
# no records should have less than 2 (first, last) and have more than one visit
x = len(df_fbg[(df_fbg["record_count"]<2) & (df_fbg["visit_count"]>1)][["subject_identifier", "record_count", "visit_count"]])

print(f"{x} rows incorrectly have more than one visit but no first/last FBG. Not fasted for second measurement??")

In [ ]:
# missing first or last FBG measurement
# now because they did not fast?
# df_fbg[(df_fbg["record_count"]<2) & (df_fbg["visit_count"]>1)][["subject_identifier", "timing", "baseline_datetime", "fbg_datetime", "record_count", "visit_count"]]



In [ ]:
x = len(df_fbg[df_fbg["visit_count"]<=1][["subject_identifier", "record_count", "visit_count"]])
y = df_fbg[df_fbg["visit_count"]<=1].subject_identifier.nunique()
print(f"{x} rows will be removed because dont have first and last FBG. {y} subjects")

In [ ]:
# this filtering drops 43 subjects who only had one schedules/unscheduled visit (baseline)
df_fbg = df_fbg[(df_fbg["record_count"]>=2) & (df_fbg["visit_count"]>1)].reset_index(drop=True)

In [ ]:
df_fbg.subject_identifier.nunique()

In [ ]:
# duration on meds / followup
# assignment
# interval between FBG measurements


In [ ]:
duplicates = df_fbg[df_fbg.duplicated(subset=["subject_identifier",  "timing"], keep=False)]
duplicates.sort_values(["subject_identifier", "timing"])

# df_fbg

In [ ]:
df_first = df_fbg[df_fbg.timing=="first"]
df_last = df_fbg[df_fbg.timing=="last"]

merged = df_first[["subject_identifier"]].merge(
    df_last[["subject_identifier"]],
    on="subject_identifier",
    how="outer",
    indicator=True
)

merged._merge.value_counts()

In [ ]:
df_fbg_pivot = df_fbg.pivot(
    index="subject_identifier",
    columns="timing",
    values="fbg_value"
).reset_index()
df_fbg_pivot.columns.name = None

# df_fbg_pivot.query("last.isna()")

In [ ]:
df_ins = read_frame(BloodResultsIns.objects.all())

In [ ]:
df_ins = (
    df_ins
    .rename(columns={"subject_visit": "subject_visit_id", "assay_datetime": "ins_datetime"})
    [["subject_visit_id", "ins_datetime", "ins_value", "ins_units", "ins_quantifier"]]
)

In [ ]:
df_ins = df_ins.merge(df_visit, on="subject_visit_id", how="left")
df_ins = df_ins[~(df_ins["subject_identifier"].isna()) & ~(df_ins["ins_datetime"].isna())].copy().reset_index(drop=True)

df_ins = df_ins.sort_values(["subject_identifier", "ins_datetime"]).reset_index(drop=True)
# ranks = df_ins.groupby("subject_identifier")["ins_datetime"].rank(method="first")
# counts = df_ins.groupby("subject_identifier")["ins_datetime"].transform("count")
# df_ins["timing"] = np.select([ranks==1, ranks==counts], ["first", "last"], default="middle")
# df_ins = df_ins[df_ins["timing"].isin(["first", "last"])]
# # duplicates = df_ins[df_ins.duplicated(subset=["subject_identifier",  "timing"], keep=False)]
# # duplicates.sort_values(["subject_identifier", "timing"])
# df_ins_pivot = df_ins.pivot(
#     index="subject_identifier",
#     columns="timing",
#     values="ins_value"
# ).reset_index()
# df_ins_pivot.columns.name = None
# df_ins_pivot.query("last.isna()")

# df_ins[["subject_identifier", "ins_datetime", "ins_value", "ins_units", "ins_quantifier", "visit_code"]]

df_ins["timing"] = np.where(df_ins["visit_code"]==1000.0, "first", "middle")
reverse_rank = df_ins.groupby("subject_identifier")["visit_code"].rank(method="first", ascending=False)
df_ins.loc[reverse_rank == 1, "timing"] = "last"
df_ins.loc[df_ins["visit_code"] == 1000.0, "timing"] = "first"

df_ins = df_ins[(df_ins["timing"].isin(["first", "last"]))][["subject_identifier", "ins_datetime", "ins_value", "ins_units", "ins_quantifier", "visit_code", "timing"]].copy().reset_index(drop=True)


In [ ]:
df  = pd.merge(df_fbg, df_ins, on=["subject_identifier", "timing"], how="left", suffixes=("_ins", "_fbg"))
# df
# df = df[~df["fbg_datetime"].isna()]

In [ ]:
print(len(df))

In [ ]:
# df = df.merge(df_visit[["subject_visit_id", "subject_identifier", "visit_code", "visit_datetime"]], on="subject_visit_id", how="left")

In [ ]:
# df[["subject_identifier", "fbg_value", "fbg_units", "fbg_datetime", "visit_code"]]
# df

In [ ]:
df = df.query("~fbg_value.isna() and ~ins_value.isna()").sort_values(by=['subject_identifier', 'timing'], ascending=True)
# df = df_sorted.drop_duplicates(subset=['subject_identifier'], keep='last').reset_index(drop=True)
df['ins_value'].astype(float)
df['fbg_value'].astype(float)
df['ins_pmol_L'] = df['ins_value'].astype(float) * 6.0

In [ ]:
df = df.merge(
    df_rando[~df_rando["subject_identifier"].isna()][["subject_identifier", "assignment", "allocated_site"]], on="subject_identifier", how="left"
).rename(columns={"allocated_site": "site"})

In [ ]:
df["site"] = df["site"].astype(int)

In [ ]:
df_export = df[["subject_identifier", "site", "assignment", "fbg_value", "ins_pmol_L", "timing"]].rename(columns={"fbg_value": "glucose", "ins_pmol_L": "insulin"}).reset_index(drop=True)

In [ ]:
if export_to_file:
    df_export.to_csv(analysis_folder / "homa2_data.csv", index=False)

In [ ]:
df.site.value_counts()

In [ ]:
(
    df_export
    .merge(
        df_rando[~df_rando["subject_identifier"].isna()][["subject_identifier", "assignment"]], on="subject_identifier", how="left"
    )
    [["subject_identifier", "assignment"]]
    .drop_duplicates()
    .assignment
    .value_counts()
)

In [ ]:
# pass the df_export to HOMA 2 calculator ...

# df_updated = pd.read_csv(analysis_folder / "homa2_data_updated.csv")
# df_updated.to_stata(
#     analysis_folder / "homa2_data_updated.dta",
#     # variable_labels=variable_labels,
#     version=118,
#     write_index=False,
# )


In [ ]:
df_export.visit_code.value_counts()

In [ ]:
from edc_model_to_dataframe.read_frame_edc import read_frame_edc
(
    read_frame_edc(BloodResultsIns.objects.all())
    .rename(columns={"subject_visit": "subject_visit_id"})
    [["subject_visit_id", "subject_identifier", "visit_code", "assay_datetime", "ins_value", "ins_units", "ins_quantifier"]]
)